# GNN-BERT Music Context EDA

This notebook inspects the verified FMA and MusicCaps manifests and the saved evaluation metrics used by the project demo.

In [1]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'src').is_dir():
        ROOT = candidate
        break
print('Project root:', ROOT)
FMA_SPLITS = ROOT / 'data' / 'splits'
MUSICCAPS_SPLITS = ROOT / 'data' / 'splits' / 'musiccaps'

Project root: D:\425 project\gnn-bert-music-context


In [2]:
def count_records(path):
    return len(json.loads(path.read_text(encoding='utf-8')))

split_rows = []
for split in ('train', 'val', 'test'):
    split_rows.append({'dataset': 'FMA', 'split': split, 'records': count_records(FMA_SPLITS / f'{split}.json')})
    split_rows.append({'dataset': 'MusicCaps', 'split': split, 'records': count_records(MUSICCAPS_SPLITS / f'{split}.json')})
pd.DataFrame(split_rows)

,dataset,split,records
0,FMA,train,80
1,MusicCaps,train,76
2,FMA,val,10
3,MusicCaps,val,9
4,FMA,test,10
5,MusicCaps,test,10


In [3]:
metric_files = sorted((ROOT / 'results').glob('*metrics.json'))
metrics = []
for path in metric_files:
    payload = json.loads(path.read_text(encoding='utf-8'))
    row = {'file': path.name}
    for key in ('macro_f1', 'micro_f1', 'auc_pr', 'caption_to_audio', 'audio_to_caption'):
        if key in payload:
            row[key] = payload[key]
    metrics.append(row)
pd.DataFrame(metrics)

,file,macro_f1,micro_f1,auc_pr,caption_to_audio,audio_to_caption
0,deam_task3_metrics.json,0.069559,0.131148,0.094574,NaN,NaN
1,expanded_fma_cnn_baseline_metrics.json,0.000000,0.000000,0.067984,NaN,NaN
2,expanded_fma_majority_baseline_metrics.json,0.000000,0.000000,0.115462,NaN,NaN
3,expanded_fma_task1_metrics.json,0.000000,0.000000,0.033385,NaN,NaN
4,expanded_fma_task2_metrics.json,0.055711,0.129496,0.110707,NaN,NaN
5,expanded_fma_task3_metrics.json,0.064227,0.123596,0.052100,NaN,NaN
6,expanded_fma_task4_metrics.json,NaN,NaN,NaN,"{'R@1': 0.2, 'R@5': 0.6, 'R@10': 1.0}","{'R@1': 0.3, 'R@5': 0.5, 'R@10': 1.0}"
7,full_fma_task2_metrics.json,0.027656,0.039481,0.018779,NaN,NaN
8,metrics.json,NaN,NaN,NaN,NaN,NaN
9,musiccaps_task4_metrics.json,NaN,NaN,NaN,"{'R@1': 0.2, 'R@5': 0.6, 'R@10': 1.0}","{'R@1': 0.2, 'R@5': 0.8, 'R@10': 1.0}"


## Interpretation

Use the split counts to verify the leakage-safe manifests and the metric table to select evidence for the presentation. The current Task 4 benchmark is the verified local MusicCaps subset, not all 5,521 source records.